# Notebook 2: BiGRU + Additive Attention Baseline
### Product Review Rating Prediction — Baseline Model Training & Evaluation

**Architecture**: FastText/GloVe Embeddings → BiGRU (2 layers) → Additive Attention Pooling → Dropout → Linear (5 classes)

> **Prerequisites**: Run `notebook1_preprocessing.ipynb` first. The `processed/` folder must exist with pre-computed arrays.

In [ ]:
!pip install -q scikit-learn matplotlib seaborn

In [ ]:
import os, pickle, random, time
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score, f1_score, confusion_matrix, classification_report
)

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

SEEDS    = [42, 123]          # run with 2 seeds, report mean ± std
NUM_CLASSES = 5
MAX_SEQ_LEN = 200

## 1. Load Preprocessed Data

In [ ]:
word2idx         = pickle.load(open('processed/word2idx.pkl', 'rb'))
embedding_matrix = np.load('processed/embedding_matrix.npy')

train_ids    = np.load('processed/train_ids.npy')
train_labels = np.load('processed/train_labels.npy')
val_ids      = np.load('processed/val_ids.npy')
val_labels   = np.load('processed/val_labels.npy')
test_ids     = np.load('processed/test_ids.npy')
test_labels  = np.load('processed/test_labels.npy')

VOCAB_SIZE = embedding_matrix.shape[0]
EMBED_DIM  = embedding_matrix.shape[1]
print(f'Vocab: {VOCAB_SIZE:,} | Embed dim: {EMBED_DIM}')
print(f'Train: {len(train_ids):,} | Val: {len(val_ids):,} | Test: {len(test_ids):,}')

## 2. Dataset & DataLoader

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, ids, labels):
        self.ids    = torch.tensor(ids,    dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):  return len(self.ids)
    def __getitem__(self, i): return self.ids[i], self.labels[i]

def make_loaders(batch_size=128, num_workers=2):
    train_ds = ReviewDataset(train_ids, train_labels)
    val_ds   = ReviewDataset(val_ids,   val_labels)
    test_ds  = ReviewDataset(test_ids,  test_labels)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, pin_memory=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers, pin_memory=True)
    return train_loader, val_loader, test_loader

## 3. Model Architecture — BiGRU + Additive Attention

In [ ]:
class AdditiveAttention(nn.Module):
    """Bahdanau-style additive attention pooling over sequence dimension."""

    def __init__(self, hidden_dim: int):
        super().__init__()
        self.attn_w = nn.Linear(hidden_dim, hidden_dim)
        self.attn_v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, h: torch.Tensor, mask: torch.Tensor | None = None):
        """
        h    : (B, T, H)
        mask : (B, T) bool, True where PAD (to be ignored)
        returns context : (B, H)
        """
        scores = self.attn_v(torch.tanh(self.attn_w(h))).squeeze(-1)  # (B, T)
        if mask is not None:
            all_pad = mask.all(dim=-1, keepdim=True)
            safe_mask = mask & (~all_pad)
            scores = scores.masked_fill(safe_mask, -1e4)
        weights = torch.softmax(scores, dim=-1)                        # (B, T)
        context = torch.bmm(weights.unsqueeze(1), h).squeeze(1)       # (B, H)
        return context, weights


class BiGRUClassifier(nn.Module):
    def __init__(
        self,
        vocab_size:    int,
        embed_dim:     int,
        hidden_dim:    int = 256,
        num_layers:    int = 2,
        num_classes:   int = 5,
        dropout:       float = 0.3,
        embedding_weights: np.ndarray | None = None,
        freeze_embed:  bool = True,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if embedding_weights is not None:
            self.embedding.weight.data.copy_(
                torch.tensor(embedding_weights, dtype=torch.float32)
            )
        if freeze_embed:
            self.embedding.weight.requires_grad_(False)

        self.gru = nn.GRU(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention = AdditiveAttention(hidden_dim * 2)  # bidirectional
        self.dropout   = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, ids: torch.Tensor):
        # ids: (B, T)
        pad_mask = (ids == 0)                     # (B, T) True = PAD
        x = self.embedding(ids)                   # (B, T, E)
        x = self.dropout(x)
        h, _ = self.gru(x)                        # (B, T, 2H)
        ctx, _ = self.attention(h, mask=pad_mask) # (B, 2H)
        out = self.classifier(self.dropout(ctx))  # (B, 5)
        return out

    def unfreeze_embedding(self):
        self.embedding.weight.requires_grad_(True)


# Sanity check
_model = BiGRUClassifier(VOCAB_SIZE, EMBED_DIM, embedding_weights=embedding_matrix)
_ids   = torch.zeros(4, MAX_SEQ_LEN, dtype=torch.long)
_out   = _model(_ids)
print('Model output shape:', _out.shape)  # (4, 5)
del _model, _ids, _out

## 4. Training Loop

In [ ]:
def evaluate(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0
    criterion  = nn.CrossEntropyLoss()
    with torch.no_grad():
        for ids, labels in loader:
            ids, labels = ids.to(device), labels.to(device)
            logits = model(ids)
            loss   = criterion(logits, labels)
            total_loss += loss.item() * len(labels)
            preds = logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())
    n    = len(all_labels)
    acc  = accuracy_score(all_labels, all_preds)
    mf1  = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    mae  = np.mean(np.abs(np.array(all_preds) - np.array(all_labels)))
    return total_loss / n, acc, mf1, mae, all_preds, all_labels


def train_bigru(
    seed: int,
    hidden_dim:   int   = 256,
    num_layers:   int   = 2,
    dropout:      float = 0.3,
    lr:           float = 1e-3,
    batch_size:   int   = 128,
    max_epochs:   int   = 15,
    patience:     int   = 3,
    grad_clip:    float = 1.0,
    num_workers:  int   = 2,
):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

    train_loader, val_loader, test_loader = make_loaders(batch_size, num_workers)

    model = BiGRUClassifier(
        VOCAB_SIZE, EMBED_DIM,
        hidden_dim=hidden_dim,
        num_layers=num_layers,
        num_classes=NUM_CLASSES,
        dropout=dropout,
        embedding_weights=embedding_matrix,
        freeze_embed=True,
    ).to(DEVICE)

    optimizer  = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    criterion  = nn.CrossEntropyLoss()
    scheduler  = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'max', patience=1, factor=0.5)

    best_mf1    = -1.0
    best_state  = None
    no_improve  = 0
    history     = {'train_loss': [], 'val_loss': [], 'val_acc': [], 'val_mf1': []}

    for epoch in range(1, max_epochs + 1):
        # Unfreeze embeddings after first epoch
        if epoch == 2:
            model.unfreeze_embedding()
            optimizer.add_param_group({'params': model.embedding.parameters(), 'lr': lr * 0.1})
            print('  [Embeddings unfrozen]')

        model.train()
        epoch_loss = 0.0
        t0 = time.time()
        for step, (ids, labels) in enumerate(train_loader):
            ids, labels = ids.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            logits = model(ids)
            loss   = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
            optimizer.step()
            epoch_loss += loss.item() * len(labels)

        train_loss = epoch_loss / len(train_loader.dataset)
        val_loss, val_acc, val_mf1, val_mae, _, _ = evaluate(model, val_loader, DEVICE)
        scheduler.step(val_mf1)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['val_mf1'].append(val_mf1)

        elapsed = time.time() - t0
        print(f'Epoch {epoch:02d} [{elapsed:.0f}s] | '
              f'Train Loss: {train_loss:.4f} | '
              f'Val Loss: {val_loss:.4f} | '
              f'Val Acc: {val_acc:.4f} | '
              f'Val MF1: {val_mf1:.4f} | '
              f'Val MAE: {val_mae:.4f}')

        if val_mf1 > best_mf1:
            best_mf1   = val_mf1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
            torch.save(best_state, f'bigru_best_seed{seed}.pt')
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'  Early stopping at epoch {epoch}.')
                break

    # Load best weights and evaluate on test
    model.load_state_dict(best_state)
    test_loss, test_acc, test_mf1, test_mae, test_preds, test_true = evaluate(model, test_loader, DEVICE)
    print(f'\nTEST — Acc: {test_acc:.4f} | MF1: {test_mf1:.4f} | MAE: {test_mae:.4f}')

    return history, test_acc, test_mf1, test_mae, test_preds, test_true, model

## 5. Run Training (2 Seeds)

In [ ]:
all_results = {}

for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'SEED {seed}')
    print('='*60)
    history, acc, mf1, mae, preds, trues, model = train_bigru(seed=seed)
    all_results[seed] = {
        'history': history,
        'acc': acc, 'mf1': mf1, 'mae': mae,
        'preds': preds, 'trues': trues,
    }

# Summary
accs = [all_results[s]['acc'] for s in SEEDS]
mf1s = [all_results[s]['mf1'] for s in SEEDS]
maes = [all_results[s]['mae'] for s in SEEDS]
print(f'\n=== FINAL RESULTS (BiGRU Baseline) ===')
print(f'Accuracy: {np.mean(accs):.4f} ± {np.std(accs):.4f}')
print(f'Macro F1: {np.mean(mf1s):.4f} ± {np.std(mf1s):.4f}')
print(f'MAE:      {np.mean(maes):.4f} ± {np.std(maes):.4f}')

## 6. Training Curves

In [ ]:
# Use results from the first seed for plotting
history = all_results[SEEDS[0]]['history']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs, history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(epochs, history['val_loss'],   label='Val Loss',   marker='s')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('BiGRU — Loss Curves'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs, history['val_acc'], label='Val Accuracy', marker='o')
axes[1].plot(epochs, history['val_mf1'], label='Val Macro F1', marker='s')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
axes[1].set_title('BiGRU — Validation Metrics'); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig('bigru_training_curves.png', dpi=150)
plt.show()

## 7. Evaluation: Confusion Matrix & Per-Class Metrics

In [ ]:
# Use best seed (highest MF1)
best_seed   = max(SEEDS, key=lambda s: all_results[s]['mf1'])
best_preds  = all_results[best_seed]['preds']
best_trues  = all_results[best_seed]['trues']

print(f'Using seed {best_seed} for detailed analysis.')
print('\nClassification Report:')
print(classification_report(
    best_trues, best_preds,
    target_names=[f'{i+1}-star' for i in range(5)]
))

# Normalized confusion matrix
cm = confusion_matrix(best_trues, best_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            xticklabels=[f'{i+1}★' for i in range(5)],
            yticklabels=[f'{i+1}★' for i in range(5)])
ax.set_xlabel('Predicted Rating'); ax.set_ylabel('True Rating')
ax.set_title('BiGRU — Normalized Confusion Matrix')
plt.tight_layout()
plt.savefig('bigru_confusion_matrix.png', dpi=150)
plt.show()

# Adjacent vs. distant confusion analysis
preds_arr = np.array(best_preds)
trues_arr = np.array(best_trues)
diffs     = np.abs(preds_arr - trues_arr)
correct   = (diffs == 0).mean()
adj1      = (diffs == 1).mean()
adj2      = (diffs == 2).mean()
dist      = (diffs >= 3).mean()
print(f'\nOrdinal Error Analysis:')
print(f'  Correct      (diff=0): {correct:.3f}')
print(f'  Adjacent     (diff=1): {adj1:.3f}')
print(f'  Near-miss    (diff=2): {adj2:.3f}')
print(f'  Distant      (diff≥3): {dist:.3f}')

## 8. Save Predictions for Cross-Model Analysis

In [ ]:
import json

summary = {
    'model': 'BiGRU',
    'seeds': SEEDS,
    'acc_mean': float(np.mean(accs)),
    'acc_std':  float(np.std(accs)),
    'mf1_mean': float(np.mean(mf1s)),
    'mf1_std':  float(np.std(mf1s)),
    'mae_mean': float(np.mean(maes)),
    'mae_std':  float(np.std(maes)),
}

with open('bigru_results.json', 'w') as f:
    json.dump(summary, f, indent=2)

np.save('bigru_test_preds.npy', np.array(best_preds))
np.save('bigru_test_trues.npy', np.array(best_trues))
print('Results saved.')
print(json.dumps(summary, indent=2))